<a href="https://colab.research.google.com/github/NeuromatchAcademy/course-content/blob/main/tutorials/W1D5_DeepLearning/student/W1D5_Intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> &nbsp; <a href="https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/NeuromatchAcademy/course-content/main/tutorials/W1D5_DeepLearning/student/W1D5_Intro.ipynb" target="_parent"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open in Kaggle"/></a>

# はじめに

## 概要

この日は、神経科学における深層学習の応用例をいくつか紹介します。イントロでは、オード・オリバが画像認識を行うために訓練された畳み込みニューラルネットワークの基本と、これらの人工ニューラルネットワークを脳の神経活動と比較する方法を解説します。3つのチュートリアルでは、神経科学で使われる深層学習の原理を3つの主要な方法で適用します：デコーディングモデル、エンコーディングモデル、そして表現類似性解析です。各チュートリアルでは、覚醒マウスの視覚野から記録された神経活動を用います。マウスには向きのあるグレーティング刺激が提示されました。チュートリアル1では、全結合の線形層のみからなるさらにシンプルなニューラルネットワークから始めます。非線形活性化関数の導入と、PyTorchと逆伝播を用いたこれらの深層ネットワークの最適化方法を説明します。視覚野の記録された神経活動から提示された視覚刺激をデコードするようネットワークを最適化します。次にチュートリアル2では、視覚タスク用ネットワークの構成要素である畳み込み層を紹介します。このチュートリアルのボーナスは、視覚刺激から神経活動へのエンコーディングモデルを適合させることです。最後にチュートリアル3では、畳み込みニューラルネットワークを最適化して向き識別タスクを実行し、表現類似性解析技術を用いて人工ニューラルネットワークの内部表現と神経活動を比較します。アウトロでは、神経活動を深層畳み込みニューラルネットワークのように扱うことの注意点を紹介し、より生物学的に妥当な深層ネットワークを作るためのアプローチを探ります。2つ目のオプションのアウトロでは、深層学習を用いて乳児の姿勢推定を行い、臨床判断に活用する例を示します。

神経科学者がより複雑な行動中のより大規模な神経集団を記録できるようになるにつれ、データ解析ツールの需要が高まっています。深層ニューラルネットワークは幅広い非線形関数を近似可能で、容易に適合できるため、大規模データのデコーディングやエンコーディングモデルの柔軟なモデル構造として適しています。W1D3では一般化線形モデル（GLM）がデコーディングおよびエンコーディングモデルとして使われました。神経活動から変数をデコードするモデルは、その変数に関して脳領域がどれだけ情報を持っているかを示します。エンコーディングモデルは視覚刺激のような入力変数から神経活動へのモデルで、脳が入力変数に対して行う変換を近似し、脳が情報をどのように表現しているか理解する助けとなります。

チュートリアル3での深層ニューラルネットワークの最終的な応用は、現在神経科学で最も一般的なもので、人工ニューラルネットワークの活動を脳活動と比較するものです。深層畳み込みニューラルネットワークは、物体認識のような視覚タスクで人間の精度を達成できる唯一のモデルタイプであるため、神経データとの比較の出発点として用いるのが理にかなっています。この比較は、チュートリアルのような集団レベルや、単一ニューロンや深層ネットワーク内の単一ユニットレベルなど、さまざまなスケールで行えます。この種の研究は、どのようなデータセットやタスクが脳を最もよく近似するニューラルネットワークを作るか（例：ニューラルタスコノミー、簡単に言えば脳活動からタスク由来表現の類似性を推測すること）、そしてそれが脳の構造や学習ルールに何を意味するかといった疑問に答える助けとなります。環境探索や報酬刺激の判定を学習するような他の複雑なタスクに対しても深層ネットワークは訓練されています。これについてはW3D4 強化学習でさらに詳しく扱います。

In [ ]:
# @title Install and import feedback gadget

!pip3 install vibecheck datatops --quiet

from vibecheck import DatatopsContentReviewContainer
def content_review(notebook_section: str):
    return DatatopsContentReviewContainer(
        "",  # No text prompt
        notebook_section,
        {
            "url": "https://pmyvdlilci.execute-api.us-east-1.amazonaws.com/klab",
            "name": "neuromatch_cn",
            "user_key": "y1x3mpx5",
        },
    ).render()


feedback_prefix = "W1D5_Intro"

## ビデオ

In [ ]:
# @markdown
from ipywidgets import widgets
from IPython.display import YouTubeVideo
from IPython.display import IFrame
from IPython.display import display


class PlayVideo(IFrame):
  def __init__(self, id, source, page=1, width=400, height=300, **kwargs):
    self.id = id
    if source == 'Bilibili':
      src = f'https://player.bilibili.com/player.html?bvid={id}&page={page}'
    elif source == 'Osf':
      src = f'https://mfr.ca-1.osf.io/render?url=https://osf.io/download/{id}/?direct%26mode=render'
    super(PlayVideo, self).__init__(src, width, height, **kwargs)


def display_videos(video_ids, W=400, H=300, fs=1):
  tab_contents = []
  for i, video_id in enumerate(video_ids):
    out = widgets.Output()
    with out:
      if video_ids[i][0] == 'Youtube':
        video = YouTubeVideo(id=video_ids[i][1], width=W,
                             height=H, fs=fs, rel=0)
        print(f'Video available at https://youtube.com/watch?v={video.id}')
      else:
        video = PlayVideo(id=video_ids[i][1], source=video_ids[i][0], width=W,
                          height=H, fs=fs, autoplay=False)
        if video_ids[i][0] == 'Bilibili':
          print(f'Video available at https://www.bilibili.com/video/{video.id}')
        elif video_ids[i][0] == 'Osf':
          print(f'Video available at https://osf.io/{video.id}')
      display(video)
    tab_contents.append(out)
  return tab_contents


video_ids = [('Youtube', 'IZvcy0Myb3M'), ('Bilibili', 'BV1sk4y1B7Ej')]
tab_contents = display_videos(video_ids, W=854, H=480)
tabs = widgets.Tab()
tabs.children = tab_contents
for i in range(len(tab_contents)):
  tabs.set_title(i, video_ids[i][0])
display(tabs)

## スライド

In [ ]:
# @markdown
from IPython.display import IFrame
link_id = "jw829"
print(f"If you want to download the slides: https://osf.io/download/{link_id}/")
IFrame(src=f"https://mfr.ca-1.osf.io/render?url=https://osf.io/{link_id}/?direct%26mode=render%26action=download%26mode=render", width=854, height=480)

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Intro")